In [ ]:
from pathlib import Path
PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Текущая рабочая папка:', PROJECT_DIR)
data_json = PROJECT_DIR / 'data.json'
if data_json.exists():
    print('data.json найден')
else:
    print('data.json не найден — если запускаете ноутбук 01, положите его в текущую папку проекта:', PROJECT_DIR)
existing = sorted(p.name for p in ARTIFACT_DIR.iterdir())
if existing:
    print(f'В artifacts/ уже есть {len(existing)} файлов/папок:')
    for name in existing:
        print(' -', name)
else:
    print('Папка artifacts/ пока пустая, она заполнится после запуска обучающих ноутбуков.')

# 07 — Gradio-приложение: сравнение моделей тональности и калькулятор индекса счастья

Приложение подгружает все обученные модели, какие реально есть в `artifacts/` (TF-IDF + LogReg/LinearSVC, RuBERT default/Optuna, xLSTM, Classic LSTM) —
Две вкладки:
1. **Сравнение моделей** — вводишь текст поста, все модели показывают свой прогноз тональности.
2. **Калькулятор индекса счастья** — считает `HI = 0.30·Сеть + 0.20·Активность + 0.50·Тональность` со штрафами за токсичные группы, «витринный» аккаунт и ночную активность.

### Проверка зависимостей

In [2]:
import importlib.util
import subprocess
import sys
REQUIRED_PACKAGES = {
    'gradio': 'gradio',
    'plotly': 'plotly',
    'transformers': 'transformers',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'joblib': 'joblib'}
missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Устанавливаются отсутствующие зависимости:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Все зависимости доступны.')

Все зависимости доступны.


## Импорты, конфигурация и загрузка моделей

In [3]:
import json
import re
import warnings
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import joblib
import gradio as gr
import plotly.graph_objects as go
from transformers import AutoModelForSequenceClassification, AutoTokenizer
PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL2NAME = {0: 'Негативный', 1: 'Позитивный', 2: 'Нейтральный'}
LABEL_EMOJI = {0: '😔', 1: '😊', 2: '😐'}
LABEL_COLOR = {0: '#ef4444', 1: '#22c55e', 2: '#94a3b8'}
TOXIC_KEYWORDS = [
    'депрессия', 'грусть', 'боль', 'слезы', 'слёзы', 'одиночество',
    'апатия', 'ненависть', 'суицид', 'смерть', 'безысходность',]
def clean_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^а-яёА-ЯЁa-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()
class Vocabulary:
    def __init__(self, token2id: Dict[str, int]):
        self.token2id = token2id
    def encode(self, text: str, max_length: int) -> List[int]:
        tokens = clean_text(text).split()
        ids = [self.token2id.get(token, 1) for token in tokens]
        ids = ids[:max_length]
        return ids + [0] * (max_length - len(ids))
    def __len__(self) -> int:
        return len(self.token2id)
class sLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gates = nn.Linear(input_dim + hidden_dim, hidden_dim * 4)
    def forward(self, x_t, state):
        h_prev, c_prev, n_prev, m_prev = state
        z_t, i_t, f_t, o_t = self.gates(torch.cat([x_t, h_prev], dim=-1)).chunk(4, dim=-1)
        z_t = torch.tanh(z_t)
        o_t = torch.sigmoid(o_t)
        log_f_t = torch.nn.functional.logsigmoid(f_t)
        m_t = torch.maximum(log_f_t + m_prev, i_t)
        i_t_stable = torch.exp(i_t - m_t)
        f_t_stable = torch.exp(log_f_t + m_prev - m_t)
        c_t = f_t_stable * c_prev + i_t_stable * z_t
        n_t = f_t_stable * n_prev + i_t_stable
        h_t = o_t * (c_t / (n_t + 1e-6))
        return h_t, c_t, n_t, m_t
class XLSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, dropout: float, num_classes: int = 3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.cell = sLSTMCell(embed_dim, hidden_dim)
        self.classifier = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))
    def forward(self, input_ids):
        embeddings = self.dropout(self.embedding(input_ids))
        batch_size, seq_len, _ = embeddings.shape
        h_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        c_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        n_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        m_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        outputs = []
        for step in range(seq_len):
            h_t, c_t, n_t, m_t = self.cell(embeddings[:, step, :], (h_t, c_t, n_t, m_t))
            outputs.append(h_t.unsqueeze(1))
        sequence_output = torch.cat(outputs, dim=1)
        lengths = (input_ids != 0).sum(dim=1).clamp(min=1)
        last_indices = (lengths - 1).view(-1, 1, 1).expand(-1, 1, self.hidden_dim)
        pooled = sequence_output.gather(dim=1, index=last_indices).squeeze(1)
        return self.classifier(pooled)
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, bidirectional, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0)
        output_dim = hidden_dim * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(output_dim), nn.Dropout(dropout), nn.Linear(output_dim, num_classes))
    def forward(self, input_ids):
        lengths = (input_ids != 0).sum(dim=1).clamp(min=1).cpu()
        embeddings = self.dropout(self.embedding(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(embeddings, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        pooled = torch.cat([h_n[-2], h_n[-1]], dim=-1) if self.lstm.bidirectional else h_n[-1]
        return self.classifier(pooled)
def load_json(path: Path) -> Optional[dict]:
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)
def make_sklearn_predictor(vectorizer, model, has_proba: bool) -> Callable[[List[str]], np.ndarray]:
    def predict(texts: List[str]) -> np.ndarray:
        clean = [clean_text(text) for text in texts]
        vectors = vectorizer.transform(clean)
        if has_proba:
            return model.predict_proba(vectors)
        scores = np.atleast_2d(model.decision_function(vectors))
        exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)
    return predict
def make_rubert_predictor(model_dir: Path) -> Callable[[List[str]], np.ndarray]:
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()
    @torch.no_grad()
    def predict(texts: List[str]) -> np.ndarray:
        encoded = tokenizer(list(texts), max_length=192, padding=True, truncation=True, return_tensors='pt')
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        probs = torch.softmax(model(**encoded).logits, dim=1)
        return probs.cpu().numpy()
    return predict
def make_recurrent_predictor(model: nn.Module, vocab: Vocabulary, max_length: int) -> Callable[[List[str]], np.ndarray]:
    model.eval()
    @torch.no_grad()
    def predict(texts: List[str]) -> np.ndarray:
        input_ids = torch.tensor(
            [vocab.encode(text, max_length) for text in texts],
            dtype=torch.long, device=DEVICE)
        probs = torch.softmax(model(input_ids), dim=1)
        return probs.cpu().numpy()
    return predict
def load_models() -> Dict[str, dict]:
    models: Dict[str, dict] = {}
    logreg_metrics = load_json(ARTIFACT_DIR / '02_logreg_metrics.json')
    logreg_path = ARTIFACT_DIR / '02_logreg_model.joblib'
    logreg_vec_path = ARTIFACT_DIR / '02_tfidf_vectorizer.joblib'
    if logreg_path.exists() and logreg_vec_path.exists():
        vectorizer = joblib.load(logreg_vec_path)
        model = joblib.load(logreg_path)
        models['TF-IDF + LogReg'] = {
            'predict': make_sklearn_predictor(vectorizer, model, has_proba=True),
            'f1': (logreg_metrics or {}).get('f1_weighted'),
            'type': 'Classical ML'}
    svc_metrics = load_json(ARTIFACT_DIR / '03_linearsvc_metrics.json')
    svc_path = ARTIFACT_DIR / '03_linearsvc_model.joblib'
    svc_vec_path = ARTIFACT_DIR / '03_tfidf_vectorizer_svc.joblib'
    if svc_path.exists() and svc_vec_path.exists():
        vectorizer = joblib.load(svc_vec_path)
        model = joblib.load(svc_path)
        models['TF-IDF + LinearSVC'] = {
            'predict': make_sklearn_predictor(vectorizer, model, has_proba=False),
            'f1': (svc_metrics or {}).get('f1_weighted'),
            'type': 'Classical ML'}
    rubert_default_metrics = load_json(ARTIFACT_DIR / '04_rubert_default_metrics.json')
    rubert_default_dir = ARTIFACT_DIR / '04_rubert_finetuned_default'
    if rubert_default_dir.exists():
        models['RuBERT (default)'] = {
            'predict': make_rubert_predictor(rubert_default_dir),
            'f1': (rubert_default_metrics or {}).get('f1_weighted'),
            'type': 'Transformer DL'}
    rubert_optuna_metrics = load_json(ARTIFACT_DIR / '05_rubert_optuna_metrics.json')
    rubert_optuna_dir = ARTIFACT_DIR / '05_rubert_finetuned_optuna'
    if rubert_optuna_dir.exists():
        models['RuBERT (Optuna)'] = {
            'predict': make_rubert_predictor(rubert_optuna_dir),
            'f1': (rubert_optuna_metrics or {}).get('f1_weighted'),
            'type': 'Transformer DL'}
    xlstm_metrics = load_json(ARTIFACT_DIR / '05_xlstm_metrics.json')
    if xlstm_metrics and Path(xlstm_metrics['state_dict_path']).exists():
        with open(xlstm_metrics['vocab_path'], 'r', encoding='utf-8') as file:
            vocab_data = json.load(file)
        vocab = Vocabulary({str(key): int(value) for key, value in vocab_data['token2id'].items()})
        model = XLSTMTextClassifier(
            vocab_size=len(vocab),
            embed_dim=int(xlstm_metrics['embed_dim']),
            hidden_dim=int(xlstm_metrics['hidden_dim']),
            dropout=float(xlstm_metrics['dropout']),
        ).to(DEVICE)
        model.load_state_dict(torch.load(xlstm_metrics['state_dict_path'], map_location=DEVICE))
        models['xLSTM-style'] = {
            'predict': make_recurrent_predictor(model, vocab, int(xlstm_metrics.get('max_length', 96))),
            'f1': xlstm_metrics.get('f1_weighted'),
            'type': 'Recurrent DL'}
    lstm_metrics = load_json(ARTIFACT_DIR / '05_lstm_metrics.json')
    if lstm_metrics and Path(lstm_metrics['state_dict_path']).exists():
        with open(lstm_metrics['vocab_path'], 'r', encoding='utf-8') as file:
            vocab_data = json.load(file)
        vocab = Vocabulary({str(key): int(value) for key, value in vocab_data['token2id'].items()})
        model = LSTMTextClassifier(
            vocab_size=len(vocab),
            embed_dim=int(lstm_metrics['embed_dim']),
            hidden_dim=int(lstm_metrics['hidden_dim']),
            num_layers=int(lstm_metrics['num_layers']),
            dropout=float(lstm_metrics['dropout']),
            bidirectional=bool(lstm_metrics['bidirectional'])).to(DEVICE)
        model.load_state_dict(torch.load(lstm_metrics['state_dict_path'], map_location=DEVICE))
        models['Classic LSTM'] = {
            'predict': make_recurrent_predictor(model, vocab, int(lstm_metrics.get('max_length', 96))),
            'f1': lstm_metrics.get('f1_weighted'),
            'type': 'Recurrent DL'}
    return models
print('Загружаем обученные модели из', ARTIFACT_DIR)
MODELS = load_models()
if MODELS:
    for name, info in sorted(MODELS.items(), key=lambda kv: kv[1]['f1'] or 0, reverse=True):
        f1_str = f"{info['f1']:.4f}" if info['f1'] is not None else 'н/д'
        print(f"  {name:<22} | {info['type']:<15} | F1={f1_str}")
else:
    print('Не найдено ни одной обученной модели. Запустите файлы 02-05 перед использованием приложения.')
MODEL_NAMES_BY_F1 = sorted(MODELS.keys(), key=lambda name: MODELS[name]['f1'] or 0, reverse=True)
BEST_MODEL_NAME = MODEL_NAMES_BY_F1[0] if MODEL_NAMES_BY_F1 else None


Загружаем обученные модели из /content/drive/MyDrive/happiness_formula/artifacts


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  RuBERT (Optuna)        | Transformer DL  | F1=0.7042
  RuBERT (default)       | Transformer DL  | F1=0.6909
  TF-IDF + LogReg        | Classical ML    | F1=0.6185
  TF-IDF + LinearSVC     | Classical ML    | F1=0.6085
  Classic LSTM           | Recurrent DL    | F1=0.5354
  xLSTM-style            | Recurrent DL    | F1=0.4733


## Логика анализа текста и расчёта индекса счастья

In [4]:
CUSTOM_CSS = """
.gradient-header {
    background: linear-gradient(135deg, #7c3aed 0%, #2563eb 100%);
    padding: 28px 32px;
    border-radius: 16px;
    color: white;
    margin-bottom: 12px;
}
.gradient-header h1 {
    margin: 0 0 6px 0;
    font-size: 28px;
}
.gradient-header p {
    margin: 0;
    opacity: 0.9;
}
.model-card {
    border-radius: 12px;
    padding: 14px 18px;
    border: 1px solid rgba(0,0,0,0.08);
    margin-bottom: 8px;
}
.happiness-summary {
    border-radius: 14px;
    padding: 18px 22px;
    background: rgba(124, 58, 237, 0.06);
    border: 1px solid rgba(124, 58, 237, 0.18);
}
"""
CLASS_ORDER = [1, 2, 0]
def analyze_text(text: str):
    if not MODELS:
        empty_fig = go.Figure()
        empty_fig.update_layout(title='Нет загруженных моделей')
        return empty_fig, 'Не найдено ни одной обученной модели в artifacts/. Запустите файлы 02-05.'
    if not text or not text.strip():
        empty_fig = go.Figure()
        empty_fig.update_layout(title='Введите текст для анализа')
        return empty_fig, 'Введите текст поста для анализа.'
    rows = []
    for name in MODEL_NAMES_BY_F1:
        probs = MODELS[name]['predict']([text])[0]
        pred = int(np.argmax(probs))
        rows.append({'name': name, 'probs': probs, 'pred': pred})
    fig = go.Figure()
    for class_id in CLASS_ORDER:
        fig.add_trace(go.Bar(
            name=LABEL2NAME[class_id],
            x=[row['name'] for row in rows],
            y=[float(row['probs'][class_id]) for row in rows],
            marker_color=LABEL_COLOR[class_id],
            text=[f"{row['probs'][class_id]:.0%}" for row in rows],
            textposition='outside'))
    fig.update_layout(
        barmode='group',
        yaxis=dict(title='Вероятность', range=[0, 1], tickformat='.0%'),
        legend=dict(orientation='h', y=1.15),
        margin=dict(t=60, b=10, l=10, r=10),
        height=380,
        template='plotly_white')
    summary_lines = ['### Предсказания по моделям\n']
    for row in rows:
        emoji = LABEL_EMOJI[row['pred']]
        label = LABEL2NAME[row['pred']]
        confidence = row['probs'][row['pred']]
        summary_lines.append(f"**{row['name']}** — {emoji} {label} (уверенность {confidence:.0%})")
    consensus = max(set(row['pred'] for row in rows), key=lambda label_id: sum(r['pred'] == label_id for r in rows))
    summary_lines.append(f"\n**Консенсус моделей:** {LABEL_EMOJI[consensus]} {LABEL2NAME[consensus]}")
    return fig, '\n\n'.join(summary_lines)
def minmax_log1p_single(value: float, assumed_max: float) -> float:
    value = max(float(value), 0.0)
    log_value = np.log1p(value)
    log_max = np.log1p(assumed_max)
    if log_max <= 0:
        return 0.0
    return float(np.clip(log_value / log_max, 0.0, 1.0))
def compute_happiness(
    friends_num, followers_num, self_posts_num, reposts_num,
    night_share, social_graph_norm, groups_text, posts_text, model_name):
    if not MODELS:
        empty_fig = go.Figure()
        return empty_fig, empty_fig, 'Не найдено ни одной обученной модели в artifacts/. Запустите файлы 02-05.'
    posts = [line.strip() for line in (posts_text or '').splitlines() if line.strip()]
    if not posts:
        empty_fig = go.Figure()
        return empty_fig, empty_fig, 'Добавьте хотя бы один пост — по нему считается тональность.'
    predict = MODELS[model_name]['predict']
    probs = predict(posts)
    preds = np.argmax(probs, axis=1)
    sentiment_score = float(np.mean(preds == 1))
    friends_norm = minmax_log1p_single(friends_num, 5000)
    total_activity = float(self_posts_num) + float(reposts_num)
    activity_norm = minmax_log1p_single(total_activity, 300)
    network_score = float(np.clip(0.65 * friends_norm + 0.35 * social_graph_norm, 0, 1))
    base_index = 0.30 * network_score + 0.20 * activity_norm + 0.50 * sentiment_score
    has_toxic = any(keyword in (groups_text or '').lower() for keyword in TOXIC_KEYWORDS)
    toxic_coef = 0.85 if has_toxic else 1.00
    follower_friend_ratio = float(followers_num) / max(float(friends_num), 1.0)
    showcase = follower_friend_ratio >= 5
    showcase_coef = 0.90 if showcase else 1.00
    insomnia_deduction = float(np.clip(0.15 * night_share, 0, 0.15))
    happiness_index = float(np.clip(base_index * toxic_coef * showcase_coef - insomnia_deduction, 0, 1))
    happiness_index = round(happiness_index, 3)
    if happiness_index >= 0.65:
        level = 'Высокий'
        level_color = '#22c55e'
    elif happiness_index >= 0.40:
        level = 'Средний'
        level_color = '#f59e0b'
    else:
        level = 'Низкий'
        level_color = '#ef4444'
    gauge_fig = go.Figure(go.Indicator(
        mode='gauge+number',
        value=happiness_index,
        number={'valueformat': '.3f', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 1]},
            'bar': {'color': level_color},
            'steps': [
                {'range': [0, 0.40], 'color': '#fee2e2'},
                {'range': [0.40, 0.65], 'color': '#fef3c7'},
                {'range': [0.65, 1.00], 'color': '#dcfce7'},
            ],
        },
        title={'text': f'Индекс счастья — уровень: {level}'}))
    gauge_fig.update_layout(height=320, margin=dict(t=60, b=10, l=30, r=30))
    breakdown_fig = go.Figure(go.Bar(
        x=['Сеть (30%)', 'Активность (20%)', 'Тональность (50%)'],
        y=[0.30 * network_score, 0.20 * activity_norm, 0.50 * sentiment_score],
        marker_color=['#2563eb', '#7c3aed', '#22c55e'],
        text=[f'{network_score:.2f}', f'{activity_norm:.2f}', f'{sentiment_score:.2f}'],
        textposition='outside'))
    breakdown_fig.update_layout(
        title='Вклад компонентов (до штрафов)',
        yaxis=dict(title='Вклад в индекс'),
        height=320,
        template='plotly_white',
        margin=dict(t=50, b=10, l=10, r=10))
    notes = [f'**Использована модель:** {model_name} | **Постов проанализировано:** {len(posts)}']
    notes.append(f'**Доля позитивных постов:** {sentiment_score:.0%}')
    if has_toxic:
        notes.append('⚠️ Найдены токсичные ключевые слова в группах — штраф ×0.85')
    if showcase:
        notes.append(f'⚠️ Похоже на витринный/коммерческий аккаунт (подписчиков/друзей = {follower_friend_ratio:.1f}) — штраф ×0.90')
    if night_share > 0:
        notes.append(f'🌙 Ночная активность: {night_share:.0%} — вычет до {insomnia_deduction:.3f}')
    return gauge_fig, breakdown_fig, '\n\n'.join(notes)

## Интерфейс Gradio

In [5]:
model_choices = MODEL_NAMES_BY_F1 if MODEL_NAMES_BY_F1 else ['Нет доступных моделей']
default_model = BEST_MODEL_NAME if BEST_MODEL_NAME else model_choices[0]
with gr.Blocks(theme=gr.themes.Soft(primary_hue='violet', secondary_hue='blue'), css=CUSTOM_CSS, title='Формула счастья VK') as demo:
    gr.HTML(
        """
        <div class="gradient-header">
            <h1>💜 Формула цифрового счастья</h1>
            <p>Сравнение моделей тональности VK-постов и расчёт индекса цифрового счастья</p>
        </div>
        """)
    if not MODELS:
        gr.Markdown(
            '## ⚠️ Не найдено ни одной обученной модели\n'
            f'Проверьте, что папка `{ARTIFACT_DIR}` содержит артефакты, и что вы запустили файлы 02-05 перед этим ноутбуком.')
    else:
        gr.Markdown(
            '**Загруженные модели:** ' + ', '.join(
                f'{name} (F1={MODELS[name]["f1"]:.3f})' if MODELS[name]['f1'] is not None else name
                for name in MODEL_NAMES_BY_F1
            )
        )
    with gr.Tabs():
        with gr.Tab('🤖 Сравнение моделей'):
            gr.Markdown('Введите текст VK-поста — каждая загруженная модель оценит его тональность.')
            with gr.Row():
                with gr.Column(scale=2):
                    text_input = gr.Textbox(
                        label='Текст поста',
                        placeholder='Например: Ну конечно, просто великолепно — всё как всегда сломалось в самый нужный момент.',
                        lines=4,
                    )
                    analyze_button = gr.Button('🔍 Проанализировать', variant='primary')
                with gr.Column(scale=3):
                    comparison_plot = gr.Plot(label='Сравнение моделей')
            predictions_summary = gr.Markdown()
            analyze_button.click(
                fn=analyze_text,
                inputs=[text_input],
                outputs=[comparison_plot, predictions_summary])
            text_input.submit(
                fn=analyze_text,
                inputs=[text_input],
                outputs=[comparison_plot, predictions_summary])
        with gr.Tab('😊 Калькулятор индекса счастья'):
            gr.Markdown(
                'Заполните профиль и вставьте посты (каждый с новой строки) — индекс считается по той же '
                'формуле, что и в финальном ноутбуке `06`: '
                '`HI = 0.30·Сеть + 0.20·Активность + 0.50·Тональность`, со штрафами за токсичные группы, '
                '«витринный» аккаунт и ночную активность.')
            with gr.Row():
                with gr.Column():
                    friends_num_input = gr.Number(label='Количество друзей', value=250, minimum=0)
                    followers_num_input = gr.Number(label='Количество подписчиков', value=120, minimum=0)
                    self_posts_num_input = gr.Number(label='Собственных постов', value=40, minimum=0)
                    reposts_num_input = gr.Number(label='Репостов', value=15, minimum=0)
                    night_share_input = gr.Slider(label='Доля постов ночью (1:00-5:00)', minimum=0, maximum=1, value=0.1, step=0.05)
                    social_graph_input = gr.Slider(
                        label='Встроенность в граф друзей (приблизительно, 0-1)',
                        minimum=0, maximum=1, value=0.3, step=0.05)
                with gr.Column():
                    groups_text_input = gr.Textbox(
                        label='Названия/описания групп пользователя',
                        placeholder='Например: фитнес-клуб, фотографии путешествий, кулинария...',
                        lines=3)
                    posts_text_input = gr.Textbox(
                        label='Посты пользователя (один пост на строку)',
                        placeholder='Сегодня был отличный день!\nУстал, но всё хорошо.\nНичего особенного не произошло.',
                        lines=6)
                    model_dropdown = gr.Dropdown(
                        label='Модель для оценки тональности',
                        choices=model_choices,
                        value=default_model)
                    compute_button = gr.Button('💜 Рассчитать индекс счастья', variant='primary')
            with gr.Row():
                gauge_plot = gr.Plot(label='Индекс счастья')
                breakdown_plot = gr.Plot(label='Вклад компонентов')
            happiness_notes = gr.Markdown()
            compute_button.click(
                fn=compute_happiness,
                inputs=[
                    friends_num_input, followers_num_input, self_posts_num_input, reposts_num_input,
                    night_share_input, social_graph_input, groups_text_input, posts_text_input, model_dropdown],
                outputs=[gauge_plot, breakdown_plot, happiness_notes])
    gr.Markdown(
        '<div style="text-align:center; opacity:0.6; margin-top:18px;">'
        'Happiness Formula · VK Digital Happiness Index · построено на TF-IDF, RuBERT, xLSTM и Classic LSTM'
        '</div>'
    )
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://295d189d243b0cb820.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
